# 02_Download_PMC_Documents

Tarea:
1. Lee `workspace.tfm_pmc.pmc_inventory`.
2. Selecciona únicamente la versión más reciente de cada artículo.
3. Excluye artículos retractados.
4. Procesa documentos con `download_status` pendiente o fallido.
5. Descarga PDF y XML/JATS cuando las URLs están disponibles.
6. Valida los archivos y calcula SHA-256.
7. Guarda los archivos en un Unity Catalog Volume.
8. Hace `MERGE` idempotente en `pmc_document_downloads`.
9. Actualiza `pmc_inventory.download_status`.
10. Registra la ejecución en `pipeline_runs`.


In [0]:
from __future__ import annotations

import hashlib
import json
import time
import uuid
import xml.etree.ElementTree as ET

from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import requests

from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql import types as T

In [0]:
# ============================================================
# Configuración
# ============================================================

CATALOG_NAME = "workspace"
SCHEMA_NAME = "tfm_pmc"

SOURCE_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_inventory"
TARGET_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pmc_document_downloads"
PIPELINE_RUNS_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.pipeline_runs"

VOLUME_NAME = "pmc_documents"
VOLUME_PATH = f"/Volumes/{CATALOG_NAME}/{SCHEMA_NAME}/{VOLUME_NAME}"

PIPELINE_NAME = "02_Download_PMC_Documents_v3"

# Durante la auditoría E2E trabajamos con un corpus pequeño.
MAX_DOCUMENTS = 20

MAX_RETRIES = 3
REQUEST_TIMEOUT_SECONDS = 120
RETRY_BACKOFF_SECONDS = 2

RUN_ID = str(uuid.uuid4())
RUN_STARTED_AT = datetime.now(timezone.utc)

print(f"Run ID: {RUN_ID}")
print(f"Maximum documents this run: {MAX_DOCUMENTS}")


Run ID: 5c040786-a802-41e3-869f-811abe61df04
Maximum documents this run: 20


In [0]:
# ============================================================
# Crear Volume y tabla de salida
# ============================================================

spark.sql(
    f"""
    CREATE VOLUME IF NOT EXISTS
    {CATALOG_NAME}.{SCHEMA_NAME}.{VOLUME_NAME}
    """
)

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS {TARGET_TABLE} (
        pmcid STRING NOT NULL,
        article_version STRING NOT NULL,
        title STRING,

        pdf_source_url STRING,
        pdf_file_path STRING,
        pdf_status STRING,
        pdf_attempts INT,
        pdf_http_status INT,
        pdf_content_type STRING,
        pdf_file_size_bytes BIGINT,
        pdf_sha256 STRING,

        xml_source_url STRING,
        xml_file_path STRING,
        xml_status STRING,
        xml_attempts INT,
        xml_http_status INT,
        xml_content_type STRING,
        xml_file_size_bytes BIGINT,
        xml_sha256 STRING,

        preferred_source STRING,
        download_successful BOOLEAN,

        inventory_run_id STRING,
        download_run_id STRING,

        error_message STRING,
        processed_at TIMESTAMP,
        updated_at TIMESTAMP
    )
    USING DELTA
    """
)

print("Volume and download table are ready.")


Volume and download table are ready.


In [0]:
# ============================================================
# Seleccionar documentos pendientes
# ============================================================

inventory_df = (
    spark.table(SOURCE_TABLE)
    .filter(F.col("is_latest_version") == True)
    .filter(
        F.coalesce(
            F.col("is_retracted"),
            F.lit(False),
        ) == False
    )
    .filter(
        F.coalesce(
            F.col("download_status"),
            F.lit("pending"),
        ).isin("pending", "failed")
    )
    .filter(
        F.col("pdf_url").isNotNull()
        | F.col("xml_url").isNotNull()
    )
    .select(
        "pmcid",
        "article_version",
        "title",
        "pdf_url",
        "xml_url",
        "inventory_run_id",
    )
    .orderBy("pmcid", "article_version")
    .limit(MAX_DOCUMENTS)
)

documents_selected = inventory_df.count()

print("Documents selected:", documents_selected)


Documents selected: 20


In [0]:
# ============================================================
# Rutas persistentes
# ============================================================

def build_article_directory(
    article_version: str,
    source_type: str,
) -> Path:

    article_directory = (
        Path(VOLUME_PATH)
        / source_type
        / article_version
    )

    article_directory.mkdir(
        parents=True,
        exist_ok=True,
    )

    return article_directory


def build_pdf_path(article_version: str) -> Path:
    return (
        build_article_directory(
            article_version,
            "pdf",
        )
        / f"{article_version}.pdf"
    )


def build_xml_path(article_version: str) -> Path:
    return (
        build_article_directory(
            article_version,
            "xml",
        )
        / f"{article_version}.xml"
    )

In [0]:
# ============================================================
# Validaciones y checksum
# ============================================================

def calculate_sha256(
    file_path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:

    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            sha256.update(chunk)

    return sha256.hexdigest()


def is_valid_pdf_file(file_path: Path) -> bool:
    if not file_path.exists() or file_path.stat().st_size == 0:
        return False

    try:
        with file_path.open("rb") as file:
            return file.read(5) == b"%PDF-"
    except OSError:
        return False


def is_valid_xml_file(file_path: Path) -> bool:
    if not file_path.exists() or file_path.stat().st_size == 0:
        return False

    try:
        ET.parse(file_path)
        return True
    except (ET.ParseError, OSError):
        return False

In [0]:
# ============================================================
# Descarga genérica con reintentos
# ============================================================

def download_file(
    source_url: str,
    output_path: Path,
    source_type: str,
    max_retries: int = MAX_RETRIES,
) -> dict[str, Any]:

    temp_path = output_path.with_suffix(
        output_path.suffix + ".part"
    )

    headers = {
        "User-Agent": (
            "PMC-TFM-Pipeline/2.0 "
            "(academic data engineering project)"
        ),
        "Accept": (
            "application/pdf,*/*"
            if source_type == "pdf"
            else "application/xml,text/xml,*/*"
        ),
    }

    last_error: Exception | None = None

    for attempt in range(1, max_retries + 1):
        try:
            if temp_path.exists():
                temp_path.unlink()

            with requests.get(
                source_url,
                headers=headers,
                timeout=REQUEST_TIMEOUT_SECONDS,
                stream=True,
                allow_redirects=True,
            ) as response:

                http_status = response.status_code
                response.raise_for_status()

                content_type = (
                    response.headers
                    .get("content-type", "")
                    .lower()
                )

                bytes_written = 0

                with temp_path.open("wb") as file:
                    for chunk in response.iter_content(
                        chunk_size=1024 * 1024
                    ):
                        if not chunk:
                            continue

                        file.write(chunk)
                        bytes_written += len(chunk)

            temp_path.replace(output_path)

            valid = (
                is_valid_pdf_file(output_path)
                if source_type == "pdf"
                else is_valid_xml_file(output_path)
            )

            if not valid:
                raise ValueError(
                    f"{source_type.upper()} validation failed."
                )

            return {
                "status": "downloaded",
                "attempts": attempt,
                "http_status": http_status,
                "content_type": content_type,
                "file_size_bytes": bytes_written,
                "sha256": calculate_sha256(output_path),
                "error_message": None,
            }

        except Exception as error:
            last_error = error

            if temp_path.exists():
                try:
                    temp_path.unlink()
                except OSError:
                    pass

            if output_path.exists():
                try:
                    output_path.unlink()
                except OSError:
                    pass

            if attempt < max_retries:
                wait_seconds = (
                    RETRY_BACKOFF_SECONDS
                    * (2 ** (attempt - 1))
                )

                print(
                    f"{source_type.upper()} attempt "
                    f"{attempt} failed. "
                    f"Retrying in {wait_seconds}s."
                )

                time.sleep(wait_seconds)

    return {
        "status": "failed",
        "attempts": max_retries,
        "http_status": None,
        "content_type": None,
        "file_size_bytes": None,
        "sha256": None,
        "error_message": str(last_error),
    }

In [0]:
def get_existing_file_result(
    output_path: Path,
    source_type: str,
) -> dict[str, Any] | None:

    valid = (
        is_valid_pdf_file(output_path)
        if source_type == "pdf"
        else is_valid_xml_file(output_path)
    )

    if not valid:
        return None

    return {
        "status": "already_exists",
        "attempts": 0,
        "http_status": None,
        "content_type": (
            "application/pdf"
            if source_type == "pdf"
            else "application/xml"
        ),
        "file_size_bytes": output_path.stat().st_size,
        "sha256": calculate_sha256(output_path),
        "error_message": None,
    }


def empty_source_result(
    status: str = "not_available",
) -> dict[str, Any]:

    return {
        "status": status,
        "attempts": 0,
        "http_status": None,
        "content_type": None,
        "file_size_bytes": None,
        "sha256": None,
        "error_message": None,
    }

In [0]:
def process_source(
    source_url: str | None,
    output_path: Path,
    source_type: str,
) -> dict[str, Any]:

    if not source_url:
        return empty_source_result()

    existing = get_existing_file_result(
        output_path,
        source_type,
    )

    if existing is not None:
        return existing

    return download_file(
        source_url=source_url,
        output_path=output_path,
        source_type=source_type,
    )


def process_inventory_record(record) -> dict[str, Any]:
    pmcid = record["pmcid"]
    article_version = record["article_version"]

    # La existencia de una URL es suficiente para determinar disponibilidad.
    pdf_url = record["pdf_url"]
    xml_url = record["xml_url"]

    pdf_path = build_pdf_path(article_version)
    xml_path = build_xml_path(article_version)

    pdf_result = process_source(
        pdf_url,
        pdf_path,
        "pdf",
    )

    xml_result = process_source(
        xml_url,
        xml_path,
        "xml",
    )

    pdf_success = pdf_result["status"] in {
        "downloaded",
        "already_exists",
    }

    xml_success = xml_result["status"] in {
        "downloaded",
        "already_exists",
    }

    if xml_success:
        preferred_source = "xml"
    elif pdf_success:
        preferred_source = "pdf"
    else:
        preferred_source = None

    errors = []

    if pdf_result["status"] == "failed":
        errors.append(f"PDF: {pdf_result['error_message']}")

    if xml_result["status"] == "failed":
        errors.append(f"XML: {xml_result['error_message']}")

    processed_at = datetime.now(timezone.utc).isoformat()

    return {
        "pmcid": pmcid,
        "article_version": article_version,
        "title": record["title"],

        "pdf_source_url": pdf_url,
        "pdf_file_path": str(pdf_path) if pdf_success else None,
        "pdf_status": pdf_result["status"],
        "pdf_attempts": pdf_result["attempts"],
        "pdf_http_status": pdf_result["http_status"],
        "pdf_content_type": pdf_result["content_type"],
        "pdf_file_size_bytes": pdf_result["file_size_bytes"],
        "pdf_sha256": pdf_result["sha256"],

        "xml_source_url": xml_url,
        "xml_file_path": str(xml_path) if xml_success else None,
        "xml_status": xml_result["status"],
        "xml_attempts": xml_result["attempts"],
        "xml_http_status": xml_result["http_status"],
        "xml_content_type": xml_result["content_type"],
        "xml_file_size_bytes": xml_result["file_size_bytes"],
        "xml_sha256": xml_result["sha256"],

        "preferred_source": preferred_source,
        "download_successful": preferred_source is not None,

        "inventory_run_id": record["inventory_run_id"],
        "download_run_id": RUN_ID,

        "error_message": " | ".join(errors) if errors else None,
        "processed_at": processed_at,
        "updated_at": processed_at,
    }


In [0]:
# ============================================================
# Procesar documentos
# ============================================================

download_results: list[dict[str, Any]] = []

for index, record in enumerate(
    inventory_df.toLocalIterator(),
    start=1,
):
    article_version = record["article_version"]

    print(
        f"[{index}/{documents_selected}] "
        f"Processing {article_version}"
    )

    result = process_inventory_record(record)

    download_results.append(result)

    print(
        f"PDF={result['pdf_status']} | "
        f"XML={result['xml_status']} | "
        f"Preferred={result['preferred_source']} | "
        f"Successful={result['download_successful']}"
    )

[1/20] Processing PMC13538227.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[2/20] Processing PMC13538279.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[3/20] Processing PMC13539412.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[4/20] Processing PMC13540131.2
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[5/20] Processing PMC13543812.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[6/20] Processing PMC13544149.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[7/20] Processing PMC13545448.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[8/20] Processing PMC13547081.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[9/20] Processing PMC13547469.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[10/20] Processing PMC13552502.1
PDF=downloaded | XML=downloaded | Preferred=xml | Successful=True
[11/20] Processing 

In [0]:
if not download_results:
    print(
        "No pending documents were found. "
        "The download stage has nothing to process."
    )
else:
    successful = sum(
        result["download_successful"]
        for result in download_results
    )

    failed = sum(
        not result["download_successful"]
        for result in download_results
    )

    xml_available = sum(
        result["xml_status"] in {
            "downloaded",
            "already_exists",
        }
        for result in download_results
    )

    print("Successful articles:", successful)
    print("Failed articles:", failed)
    print("Articles with usable XML:", xml_available)

Successful articles: 20
Failed articles: 0
Articles with usable XML: 20


In [0]:
download_schema = T.StructType([
    T.StructField("pmcid", T.StringType(), False),
    T.StructField("article_version", T.StringType(), False),
    T.StructField("title", T.StringType(), True),

    T.StructField("pdf_source_url", T.StringType(), True),
    T.StructField("pdf_file_path", T.StringType(), True),
    T.StructField("pdf_status", T.StringType(), True),
    T.StructField("pdf_attempts", T.IntegerType(), True),
    T.StructField("pdf_http_status", T.IntegerType(), True),
    T.StructField("pdf_content_type", T.StringType(), True),
    T.StructField("pdf_file_size_bytes", T.LongType(), True),
    T.StructField("pdf_sha256", T.StringType(), True),

    T.StructField("xml_source_url", T.StringType(), True),
    T.StructField("xml_file_path", T.StringType(), True),
    T.StructField("xml_status", T.StringType(), True),
    T.StructField("xml_attempts", T.IntegerType(), True),
    T.StructField("xml_http_status", T.IntegerType(), True),
    T.StructField("xml_content_type", T.StringType(), True),
    T.StructField("xml_file_size_bytes", T.LongType(), True),
    T.StructField("xml_sha256", T.StringType(), True),

    T.StructField("preferred_source", T.StringType(), True),
    T.StructField("download_successful", T.BooleanType(), False),

    T.StructField("inventory_run_id", T.StringType(), True),
    T.StructField("download_run_id", T.StringType(), False),

    T.StructField("error_message", T.StringType(), True),
    T.StructField("processed_at", T.StringType(), False),
    T.StructField("updated_at", T.StringType(), False),
])


In [0]:
# ============================================================
# Crear DataFrame de resultados
# ============================================================

downloads_df = None

if download_results:
    downloads_df = (
        spark.createDataFrame(
            download_results,
            schema=download_schema,
        )
        .withColumn(
            "processed_at",
            F.to_timestamp("processed_at"),
        )
        .withColumn(
            "updated_at",
            F.to_timestamp("updated_at"),
        )
    )


In [0]:
# ============================================================
# Validaciones básicas
# ============================================================

if downloads_df is not None:
    summary = downloads_df.agg(
        F.count("*").alias("total_articles"),
        F.sum(F.col("download_successful").cast("int")).alias("successful_articles"),
        F.sum((~F.col("download_successful")).cast("int")).alias("failed_articles"),
        F.sum(F.col("pdf_status").isin("downloaded", "already_exists").cast("int")).alias("usable_pdfs"),
        F.sum(F.col("xml_status").isin("downloaded", "already_exists").cast("int")).alias("usable_xmls"),
    ).first()

    print(
        f"Processed={summary['total_articles']} | "
        f"Successful={summary['successful_articles']} | "
        f"Failed={summary['failed_articles']} | "
        f"PDF={summary['usable_pdfs']} | "
        f"XML={summary['usable_xmls']}"
    )


Processed=20 | Successful=20 | Failed=0 | PDF=20 | XML=20


In [0]:
# ============================================================
# MERGE en pmc_document_downloads
# ============================================================

if downloads_df is not None:
    target_delta = DeltaTable.forName(
        spark,
        TARGET_TABLE,
    )

    (
        target_delta.alias("target")
        .merge(
            downloads_df.alias("source"),
            """
            target.pmcid = source.pmcid
            AND target.article_version = source.article_version
            """,
        )
        .whenMatchedUpdate(
            set={
                "title": "source.title",

                "pdf_source_url": "source.pdf_source_url",
                "pdf_file_path": "source.pdf_file_path",
                "pdf_status": "source.pdf_status",
                "pdf_attempts": "source.pdf_attempts",
                "pdf_http_status": "source.pdf_http_status",
                "pdf_content_type": "source.pdf_content_type",
                "pdf_file_size_bytes": "source.pdf_file_size_bytes",
                "pdf_sha256": "source.pdf_sha256",

                "xml_source_url": "source.xml_source_url",
                "xml_file_path": "source.xml_file_path",
                "xml_status": "source.xml_status",
                "xml_attempts": "source.xml_attempts",
                "xml_http_status": "source.xml_http_status",
                "xml_content_type": "source.xml_content_type",
                "xml_file_size_bytes": "source.xml_file_size_bytes",
                "xml_sha256": "source.xml_sha256",

                "preferred_source": "source.preferred_source",
                "download_successful": "source.download_successful",

                "inventory_run_id": "source.inventory_run_id",
                "download_run_id": "source.download_run_id",
                "error_message": "source.error_message",
                "processed_at": "source.processed_at",
                "updated_at": "source.updated_at",
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    )

    print("Download results MERGE completed.")


Download results MERGE completed.


In [0]:
# ============================================================
# Actualizar estado del inventario
# ============================================================

if downloads_df is not None:
    inventory_delta = DeltaTable.forName(
        spark,
        SOURCE_TABLE,
    )

    status_df = (
        downloads_df
        .select(
            "pmcid",
            "article_version",

            F.when(
                F.col("download_successful"),
                F.lit("completed"),
            )
            .otherwise(
                F.lit("failed")
            )
            .alias(
                "new_download_status"
            ),

            "error_message",
            "updated_at",
        )
    )

    (
        inventory_delta.alias("target")
        .merge(
            status_df.alias("source"),
            """
            target.pmcid = source.pmcid
            AND target.article_version = source.article_version
            """,
        )
        .whenMatchedUpdate(
            set={
                "download_status": "source.new_download_status",
                "error_message": "source.error_message",
                "updated_at": "source.updated_at",
            }
        )
        .execute()
    )

    print(
        "pmc_inventory download statuses updated."
    )

pmc_inventory download statuses updated.


In [0]:
# ============================================================
# Registrar ejecución
# ============================================================

RUN_COMPLETED_AT = datetime.now(
    timezone.utc
)

records_processed = len(download_results)

records_successful = sum(
    result["download_successful"]
    for result in download_results
)

records_failed = sum(
    not result["download_successful"]
    for result in download_results
)

xml_successful = sum(
    result["xml_status"] in {
        "downloaded",
        "already_exists",
    }
    for result in download_results
)

pdf_successful = sum(
    result["pdf_status"] in {
        "downloaded",
        "already_exists",
    }
    for result in download_results
)

run_row = spark.createDataFrame(
    [
        (
            RUN_ID,
            PIPELINE_NAME,
            (
                "completed"
                if records_failed == 0
                else "completed_with_errors"
            ),
            RUN_STARTED_AT,
            RUN_COMPLETED_AT,
            int(MAX_DOCUMENTS),
            int(documents_selected),
            int(records_processed),
            int(records_successful),
            0,
            int(records_failed),
            json.dumps({
                "volume_path": VOLUME_PATH,
                "source_table": SOURCE_TABLE,
                "target_table": TARGET_TABLE,
                "pdf_successful": pdf_successful,
                "xml_successful": xml_successful,
                "preferred_extraction_source": "xml",
                "pdf_available_for_extraction": True,
            }),
            None,
        )
    ],
    schema=T.StructType([
        T.StructField("run_id", T.StringType(), False),
        T.StructField("pipeline_name", T.StringType(), False),
        T.StructField("run_status", T.StringType(), False),
        T.StructField("started_at", T.TimestampType(), False),
        T.StructField("completed_at", T.TimestampType(), True),
        T.StructField("records_requested", T.LongType(), True),
        T.StructField("records_found", T.LongType(), True),
        T.StructField("records_processed", T.LongType(), True),
        T.StructField("records_inserted", T.LongType(), True),
        T.StructField("records_updated", T.LongType(), True),
        T.StructField("records_failed", T.LongType(), True),
        T.StructField("execution_metadata", T.StringType(), True),
        T.StructField("error_message", T.StringType(), True),
    ]),
)

run_row.write.mode(
    "append"
).saveAsTable(
    PIPELINE_RUNS_TABLE
)

print(f"Pipeline run registered: {RUN_ID}")

Pipeline run registered: 5c040786-a802-41e3-869f-811abe61df04
